In [ ]:
import re
from typing import List, Optional

from pypdf import PdfReader

In [ ]:
def extract_text_from_pdf(
    file_path: str, page_numbers: Optional[List[int]] = None
) -> str:
    try:
        reader = PdfReader(file_path)
        total_pages = len(reader.pages)

        if page_numbers is None:
            pages_to_extract = range(1, total_pages + 1)
        else:
            pages_to_extract = []
            for page_num in page_numbers:
                if page_num < 1 or page_num > total_pages:
                    raise ValueError(
                        f"Page number {page_num} is out of range. PDF has {total_pages} page(s)."
                    )
                pages_to_extract.append(page_num)

        extracted_text = []
        for page_num in pages_to_extract:
            page = reader.pages[page_num - 1]
            text = page.extract_text()
            if text:
                extracted_text.append(text)

        return "\n\n".join(extracted_text)

    except FileNotFoundError:
        raise FileNotFoundError(f"PDF file not found: {file_path}")
    except Exception as e:
        raise Exception(f"Error extracting text from PDF: {str(e)}")

In [ ]:
# Extract pages 3 (beaches) and 4 (ports) separately
beach_text_raw = extract_text_from_pdf(
    "pavillon_bleu_2018_201806.pdf",
    page_numbers=[3],
)
port_text_raw = extract_text_from_pdf(
    "pavillon_bleu_2018_201806.pdf",
    page_numbers=[4],
)

print("=== PAGE 3 (BEACHES) ===")
print(beach_text_raw[:500])
print("\n=== PAGE 4 (PORTS) ===")
print(port_text_raw[:500])

In [ ]:
# Regions for 2018
regions = [
    "AUVERGNE-RHÔNE-ALPES",
    "BOURGOGNE-FRANCHE-COMTÉ",
    "BRETAGNE",
    "CENTRE-VAL DE LOIRE",
    "CORSE",
    "GRAND-EST",
    "HAUTS-DE-FRANCE",
    "ÎLE-DE-FRANCE",
    "NORMANDIE",
    "NOUVELLE-AQUITAINE",
    "OCCITANIE",
    "OUTRE-MER",
    "PAYS DE LA LOIRE",
    "PROVENCE-ALPES-CÔTE D'AZUR",
    # Variants after PDF space collapsing:
    "CENTRE-VALDELOIRE",
    "PAYSDELALOIRE",
    "PROVENCE-ALPESCÔTED'AZUR",
    "PROVENCE-ALPESCÔTESDAZUR",
    "PROVENCE-ALPES-CÔTESD'AZUR",
]

REGION_NAME_FIX = {
    "CENTRE-VALDELOIRE": "CENTRE-VAL DE LOIRE",
    "PAYSDELALOIRE": "PAYS DE LA LOIRE",
    "PROVENCE-ALPESCÔTED'AZUR": "PROVENCE-ALPES-CÔTE D'AZUR",
    "PROVENCE-ALPESCÔTESDAZUR": "PROVENCE-ALPES-CÔTE D'AZUR",
    "PROVENCE-ALPES-CÔTESD'AZUR": "PROVENCE-ALPES-CÔTE D'AZUR",
}

In [ ]:
# Departments for 2018 - format is "DD NAME" (no dot after number)
departements = [
    "01 AIN",
    "02 AISNE",
    "03 ALLIER",
    "04 ALPES-DE-HAUTE-PROVENCE",
    "05 HAUTES-ALPES",
    "06 ALPES-MARITIMES",
    "07 ARDÈCHE",
    "08 ARDENNES",
    "10 AUBE",
    "11 AUDE",
    "12 AVEYRON",
    "13 BOUCHES-DU-RHÔNE",
    "14 CALVADOS",
    "15 CANTAL",
    "17 CHARENTE-MARITIME",
    "19 CORRÈZE",
    "21 CÔTE-D'OR",
    "22 CÔTES-D'ARMOR",
    "24 DORDOGNE",
    "25 DOUBS",
    "26 DRÔME",
    "29 FINISTÈRE",
    "30 GARD",
    "31 HAUTE-GARONNE",
    "32 GERS",
    "33 GIRONDE",
    "34 HÉRAULT",
    "35 ILLE-ET-VILAINE",
    "36 INDRE",
    "37 INDRE-ET-LOIRE",
    "39 JURA",
    "40 LANDES",
    "41 LOIR-ET-CHER",
    "43 HAUTE-LOIRE",
    "44 LOIRE-ATLANTIQUE",
    "45 LOIRET",
    "46 LOT",
    "47 LOT-ET-GARONNE",
    "48 LOZÈRE",
    "50 MANCHE",
    "51 MARNE",
    "52 HAUTE-MARNE",
    "54 MEURTHE-ET-MOSELLE",
    "55 MEUSE",
    "56 MORBIHAN",
    "57 MOSELLE",
    "58 NIÈVRE",
    "59 NORD",
    "60 OISE",
    "62 PAS-DE-CALAIS",
    "63 PUY-DE-DÔME",
    "64 PYRÉNÉES-ATLANTIQUES",
    "66 PYRÉNÉES-ORIENTALES",
    "67 BAS-RHIN",
    "68 HAUT-RHIN",
    "71 SAÔNE-ET-LOIRE",
    "72 SARTHE",
    "73 SAVOIE",
    "74 HAUTE-SAVOIE",
    "76 SEINE-MARITIME",
    "77 SEINE-ET-MARNE",
    "78 YVELINES",
    "80 SOMME",
    "81 TARN",
    "82 TARN-ET-GARONNE",
    "83 VAR",
    "85 VENDÉE",
    "86 VIENNE",
    "87 HAUTE-VIENNE",
    "95 VAL-D'OISE",
    # OUTRE-MER
    "97 LA-RÉUNION",
    "97 MARTINIQUE",
    "98 POLYNÉSIE-FRANÇAISE",
    "98 NOUVELLE-CALÉDONIE",
    # Corsica
    "2A CORSE-DU-SUD",
    "2B HAUTE-CORSE",
    "2ACORSE-DU-SUD",
    "2BHAUTE-CORSE",
    "97LA-RÉUNION",
    "97MARTINIQUE",
    "98POLYNÉSIE-FRANÇAISE",
    "98NOUVELLE-CALÉDONIE",
]

In [ ]:
trim_strings = (
    "DE LA MER À LA TERRE",
    "PRINTEMPS 2018",
    "PALMARÈS 2018",
    "LABELLISATION 2018-2019",
    "PLAGES LABELLISÉES",
    "PORTS LABELLISÉS",
    "PLAGES LABELLÉES",
    "PORTS LABELLÉS",
    "PLAGES  LABELLISÉES",
    "PORTS  LABELLISÉS",
    "PLAGES  LABELLÉES",
    "PORTS  LABELLÉS",
    "PLAGESLABELLISÉES",
    "PORTSLABELLISÉS",
    "PLAGESLABELLÉES",
    "PORTSLABELLÉS",
    "NOMBRE DE COMMUNES LABELLISÉES PAVILLON BLEU PAR RÉGION",
    "NOMBRE DE PORTS LABELLISÉS PAVILLON BLEU PAR RÉGION",
    "SUR 186 COMMUNES399",
    "107",
    "NEW",
    "Retrouvez les sites labellisés Pavillon Bleu proches de chez vous sur l'application Iphone",
    "Une seconde vie pour les drapeaux Pavillon Bleu",
)

BEACH_KEYWORDS = [
    "plage",
    "base",
    "plan",
    "lac",
    "étang",
    "etang",
    "baignade",
    "complexe",
    "camping",
    "piscine",
    "grande",
    "grand'",
    "poste",
    "ponton",
    "espace",
    "secteur",
    "face",
    "plaisanciers",
    "plaisance",
]

In [ ]:
def split_by_pattern_list(pattern_list, text):
    pattern = "|".join(re.escape(item) for item in pattern_list)
    pattern_with_capture = f"({pattern})"
    return re.split(pattern_with_capture, text)

In [ ]:
def preprocess_pdf_text(raw_text):
    text = raw_text

    # Join lines first so trim strings can match across newlines
    text = "".join(text.split("\n")).strip()
    # Remove statistics garbage (e.g. "1BRETAGNE12")
    text = re.sub(r"\b\d+[A-Z\u00C0-\u00DC\u00C9\u00C8]{2,}\d+\b", "", text)
    # Insert OUTRE-MER region marker before Outre-Mer departments
    # These appear after LABELLISATION marker without a region header
    for marker in ["97 LA RÉUNION", "97 MARTINIQUE", "98 POL YNÉSIE FRANÇAISE", "98 POLYNÉSIE FRANÇAISE", "98 NOUVELLE-CALÉDONIE"]:
        text = text.replace(marker, f"OUTRE-MER {marker}")

    # Protect multi-word department names before collapsing spaces
    # (replace spaces with hyphens so they survive the space collapse)
    dept_protections = [
        ("97 LA RÉUNION", "97 LA-RÉUNION"),
        ("97 MARTINIQUE", "97 MARTINIQUE"),
        ("98 POL YNÉSIE FRANÇAISE", "98 POLYNÉSIE-FRANÇAISE"),
        ("98 POLYNÉSIE FRANÇAISE", "98 POLYNÉSIE-FRANÇAISE"),
        ("98 NOUVELLE-CALÉDONIE", "98 NOUVELLE-CALÉDONIE"),
        ("2A CORSE DU SUD", "2A CORSE-DU-SUD"),
        ("2B HAUTE CORSE", "2B HAUTE-CORSE"),
    ]
    for raw, clean in dept_protections:
        # Skip if the raw form matches the clean form
        if raw != clean:
            text = text.replace(raw, clean)

    # Collapse spurious PDF spaces within uppercase words
    # Step 1: Collapse spaces between two uppercase letters
    text = re.sub(r'(?<=[A-Z\u00C0-\u00DC]) (?=[A-Z\u00C0-\u00DC])', '', text)
    # Step 1b: Collapse space after hyphen (PDF line-break hyphenation)
    text = re.sub(r'(?<=[a-zA-Z]-) ', '', text)
    # Collapse spurious PDF spaces within uppercase words
    # Step 1: Collapse space before hyphen when hyphen followed by uppercase
    text = re.sub(r' (?=-[A-Z\u00C0-\u00DC])', '', text)
    # Remove unwanted strings
    for unwanted in trim_strings:
        text = text.replace(unwanted, "")

    # Step 3: Collapse space after hyphen when preceded by uppercase
    text = re.sub(r'(?<=[A-Z\u00C0-\u00DC]-) ', '', text)
    # Step 4: Collapse space after apostrophe in uppercase context
    text = re.sub(r"['\u2019] (?=[A-Z\u00C0-\u00DC])", "'", text)
    # Step 5: Collapse spaces within digits
    text = re.sub(r'(?<=\d) (?=\d)', '', text)

    # Collapse multiple spaces
    text = re.sub(r"\s+", " ", text)

    return text

def extract_commune_beaches(town_entry):
    items = [i.strip() for i in town_entry.split(",")]
    if not items or not items[0]:
        return None, []

    first = items[0]
    words = first.split()

    starts_with_article = words[0].lower().strip("'") in ("le", "la", "les")
    skip = 2 if starts_with_article else 1

    split_idx = None
    for i in range(skip, len(words)):
        w = words[i].lower().replace("'", "")
        for kw in BEACH_KEYWORDS:
            if w.startswith(kw) or w == kw:
                split_idx = i
                break
        if split_idx is not None:
            break

    if split_idx is not None:
        commune = " ".join(words[:split_idx])
        first_beach = " ".join(words[split_idx:])
    else:
        if starts_with_article and len(words) >= 2:
            if (
                len(words) >= 4
                and words[1] == "Grande"
                and words[2].startswith("Motte")
            ):
                commune = " ".join(words[:3])
                first_beach = " ".join(words[3:])
            else:
                commune = " ".join(words[:2])
                first_beach = " ".join(words[2:])
        else:
            commune = words[0]
            first_beach = " ".join(words[1:])

    beaches = []
    if first_beach:
        beaches.append(first_beach)
    beaches.extend(items[1:])
    beaches = [b.strip().rstrip(".") for b in beaches if b.strip()]

    return commune, beaches

def dept_number_and_name(dept_code):
    parts = dept_code.split(" ", 1)
    if len(parts) == 2:
        return parts[0].strip(), parts[1].strip()
    return "", dept_code

In [ ]:
# Preprocess both texts
beach_text = preprocess_pdf_text(beach_text_raw)
port_text = preprocess_pdf_text(port_text_raw)

# Department-to-region overrides
DEPT_TO_REGION = {
    "04 ALPES-DE-HAUTE-PROVENCE": "PROVENCE-ALPES-CÔTE D'AZUR",
    "05 HAUTES-ALPES": "PROVENCE-ALPES-CÔTE D'AZUR",
    "06 ALPES-MARITIMES": "PROVENCE-ALPES-CÔTE D'AZUR",
    "13 BOUCHES-DU-RHÔNE": "PROVENCE-ALPES-CÔTE D'AZUR",
    "83 VAR": "PROVENCE-ALPES-CÔTE D'AZUR",
}

print("=== BEACH TEXT (preprocessed) ===")
print(beach_text[:2000])
print("\n=== PORT TEXT (preprocessed) ===")
print(port_text[:2000])

In [ ]:
import csv
import unicodedata
import re as re_module

YEAR = 2018
OUTPUT_FILE = "terragir_officiel_2018.csv"

regions_nfc = [unicodedata.normalize("NFC", r) for r in regions]
departements_nfc = [unicodedata.normalize("NFC", d) for d in departements]


def parse_source(source_text, records_list, is_port=False):
    if not source_text.strip():
        return

    splitted = split_by_pattern_list(regions_nfc, source_text)
    splitted = splitted[1:]

    for i in range(1, len(splitted), 2):
        region = splitted[i - 1].strip()
        region = unicodedata.normalize("NFC", region)
        if region in REGION_NAME_FIX:
            region = REGION_NAME_FIX[region]

        region_content = splitted[i].strip()
        region_content = unicodedata.normalize("NFC", region_content)
        region_content = re_module.sub(r"\s+", " ", region_content)

        splitted_dpt = split_by_pattern_list(departements_nfc, region_content)
        splitted_dpt = splitted_dpt[1:]

        current_dept = None
        for item in splitted_dpt:
            item = item.strip()
            if not item:
                continue
            if item in departements_nfc:
                current_dept = item
                if current_dept in DEPT_TO_REGION:
                    region = DEPT_TO_REGION[current_dept]
                continue
            if current_dept is None:
                continue

            # Split by pipe to get individual entries
            parts = [p.strip() for p in item.split("|") if p.strip()]
            if not parts:
                continue

            dept_num, dept_name = dept_number_and_name(current_dept)

            for town_raw in parts:
                if "(suite)" in town_raw.lower():
                    continue

                if is_port:
                    commune, places = extract_port_commune(town_raw)
                    if commune and places and any(c.isalpha() for c in commune):
                        records_list.append(
                            {
                                "year": YEAR,
                                "commune": commune.upper(),
                                "department_name": dept_name,
                                "department_number": dept_num,
                                "region": region,
                                "beach_flag": 0,
                                "port_flag": 1,
                                "places": places,
                            }
                        )
                else:
                    if town_raw.isdigit():
                        continue
                    commune, beaches = extract_commune_beaches(town_raw)
                    if commune and beaches and any(c.isalpha() for c in commune):
                        records_list.append(
                            {
                                "year": YEAR,
                                "commune": commune.upper(),
                                "department_name": dept_name,
                                "department_number": dept_num,
                                "region": region,
                                "beach_flag": 1,
                                "port_flag": 0,
                                "places": ", ".join(beaches),
                            }
                        )


def extract_port_commune(town_raw):
    """Extract commune name from a port entry."""
    full_entry = town_raw.strip()
    if not full_entry:
        return None, ""

    _town = full_entry.upper().strip()

    _port_prefixes = [
        "PORT DE PLAISANCE DE ",
        "PORT DE PLAISANCE ",
        "PORT FLUVIAL DES ",
        "PORT FLUVIAL DE ",
        "PORT FLUVIAL ",
        "PORT DE ",
        "PORT DES ",
        "PORT ",
        "BASSIN DE PLAISANCE DE ",
        "BASSIN DE PLAISANCE ",
        "BASSIN DE ",
        "BASSIN ",
        "HALTE FLUVIALE DE ",
        "HALTE FLUVIALE ",
        "HALTE DE ",
        "HALTE ",
        "RELAIS NAUTIQUE DE L'",
        "RELAIS NAUTIQUE DE ",
        "RELAIS NAUTIQUE D'",
        "RELAIS NAUTIQUE ",
        "STATION NAUTIQUE DE ",
        "NOUVEAU PORT DES ",
        "NOUVEAU PORT DE ",
        "NOUVEAU PORT ",
        "VIEUX PORT DES ",
        "VIEUX PORT DE ",
        "VIEUX PORT ",
        "CALANQUE DE ",
        "CNTL DE ",
        "SOCIÉTÉ NAUTIQUE DE ",
        "Y ACHTING CLUB ",
        "MARINA ",
    ]
    for _pfx in sorted(_port_prefixes, key=len, reverse=True):
        if _town.startswith(_pfx) and _town != _pfx:
            _town = _town[len(_pfx):]
            break

    _town = re.sub(r"\s*\(.*?\)", "", _town).strip()
    for _prep in ["DE LA ", "DE L'", "DU ", "DES ", "DE ", "D'"]:
        if _town.startswith(_prep):
            _town = _town[len(_prep):]
            break

    if _town:
        for _sep in [" DE ", " DES "]:
            if _sep in _town:
                _town = _town.rsplit(_sep, 1)[-1].strip()
                break

    return _town if _town else full_entry.upper(), full_entry


records = []

parse_source(beach_text, records, is_port=False)
parse_source(port_text, records, is_port=True)

with open(OUTPUT_FILE, "w", newline="", encoding="utf-8") as f:
    fields = [
        "year", "commune", "department_name", "department_number",
        "region", "beach_flag", "port_flag", "places",
    ]
    writer = csv.DictWriter(f, fieldnames=fields)
    writer.writeheader()
    writer.writerows(records)

print(f"Done! Wrote {len(records)} records to {OUTPUT_FILE}")

In [ ]:
# === CLEANUP: fix known PDF extraction artifacts ===

# Manual commune overrides for known port entries
PORT_COMMUNE_OVERRIDES = {
    "SAINT-SAUVEUR DE T OULOUSE": "TOULOUSE",
    "T OULOUSE": "TOULOUSE",
    "SAINT-ANGE-DU-BARCARÈS": "PORT-BARCARÈS",
    "CAMILLE RAYON DU GOLFE JUAN": "GOLFE-JUAN",
    "PIERRE CANTO": "CANNES",
    "Y ACHTING CLUB POINTE ROUGE": "MARSEILLE",
    "PORT SAINT-GERVAIS DE FOS SUR MER": "FOS-SUR-MER",
    "PORT MIOU": "CASSIS",
    "NAPOLEON": "MARTIGUES",
    "PORT NAPOLEON": "MARTIGUES",
    "CASSIS GTC": "CASSIS",
    "PRÈS-DU-HEM LILLE MÉTROPOLE": "LILLE MÉTROPOLE",
    "LA PORTE DU HAINAUT": "PORTE-DU-HAINAUT",
    "D'ÉTAPLES": "ÉTAPLES",
    "FRANCE": "NANCY",
    "SAINT-JUST": "SAINT-JUST D'ARDÈCHE",
    "LA LOIRE1BRETAGNE12": "SAINT-VALÉRY-SUR-SOMME",
    "DÉPARTEMENTAL VENDRES EN DOMITIENNE": "VENDRES",
}

for rec in records:
    commune = rec["commune"]
    places = rec["places"]
    dept = rec["department_number"]

    # Apply manual commune overrides
    if commune in PORT_COMMUNE_OVERRIDES:
        rec["commune"] = PORT_COMMUNE_OVERRIDES[commune]

    # 1. Fix "PORT" -> "PORT-DE-BOUC" in Bouches-du-Rhône
    if commune == "PORT" and dept == "13":
        rec["commune"] = "PORT-DE-BOUC"
        rec["places"] = "Bottai"
        continue

    if commune == "BOUC" and dept == "13":
        rec["commune"] = "PORT-DE-BOUC"
        rec["places"] = f"Port de Plaisance de {rec['places']}"
        continue

    # 2. Fix Bora Bora
    if commune == "BORA" and dept == "98":
        rec["commune"] = "BORA BORA"
        parts = [p.strip() for p in rec["places"].split(",")]
        if parts and parts[0].startswith("Bora "):
            parts[0] = parts[0][5:]
        rec["places"] = ", ".join(parts)
        continue

    # 3. Fix commune deduplication
    words = commune.split()
    if len(words) >= 2:
        first_word = words[0]
        last_word = words[-1]

        if first_word == last_word:
            rec["commune"] = " ".join(words[:-1])
            places_parts = [p.strip() for p in rec["places"].split(",")]
            if places_parts:
                places_parts[0] = f"{last_word.capitalize()} {places_parts[0]}"
            rec["places"] = ", ".join(places_parts)

        elif "-" in first_word and last_word in first_word.split("-"):
            rec["commune"] = " ".join(words[:-1])
            places_parts = [p.strip() for p in rec["places"].split(",")]
            if places_parts:
                places_parts[0] = f"{last_word.capitalize()} {places_parts[0]}"
            rec["places"] = ", ".join(places_parts)

    # 4. Fix trailing articles ("LA", "LE", "LES") in commune
    words = rec["commune"].split()
    if len(words) >= 2 and words[-1] in ("LA", "LE", "LES"):
        article = words[-1].title()
        rec["commune"] = " ".join(words[:-1])
        places_parts = [p.strip() for p in rec["places"].split(",")]
        if places_parts:
            places_parts[0] = f"{article} {places_parts[0]}"
        rec["places"] = ", ".join(places_parts)

# 5. Strip region-name artifacts from places
REGION_NAMES = [
    "PROVENCE-ALPES-CÔTE D'AZUR", "AUVERGNE-RHÔNE-ALPES",
    "BOURGOGNE-FRANCHE-COMTÉ", "BRETAGNE", "CENTRE-VAL DE LOIRE",
    "CORSE", "GRAND-EST", "HAUTS-DE-FRANCE", "ÎLE-DE-FRANCE",
    "NORMANDIE", "NOUVELLE-AQUITAINE", "OCCITANIE", "OUTRE-MER",
    "PAYS DE LA LOIRE",
]
for rec in records:
    places = rec["places"]
    for name in REGION_NAMES:
        places = places.replace(name, "")
    places = re.sub(r"\s+\d{1,2}\b", "", places)
    places = places.strip().rstrip(",").strip()
    rec["places"] = places

# 6. Port artifact detection + reclassification
PORT_ARTIFACT_COMMUNES = {"PORT", "HALTE", "RELAIS", "BASSIN", "STATION", "NAUTIC'HAM"}
PORT_KEYWORDS = [
    "port de plaisance", "Port de plaisance", "Port de Plaisance",
    "halte fluviale", "Halte Fluviale",
    "Port fluvial", "Marina",
    "bassin de plaisance", "Bassin de plaisance", "Bassin de Plaisance",
    "Nouveau Port", "Vieux Port",
    "Calanque de", "CNTL de", "Société Nautique",
]

dept_code_pattern = re.compile(r"\d{1,2}[A-Z]?\.?\s*[A-Z\u00C9\u00C8\u00CA\u00CB\u00C0\u00C2\u00CE\u00CF\u00D4\u00DB\u00DC\u00C7\u2019' -]+")

DEPT_REGION_MAP = {
    "01": "AUVERGNE-RHÔNE-ALPES", "02": "HAUTS-DE-FRANCE",
    "03": "AUVERGNE-RHÔNE-ALPES", "04": "PROVENCE-ALPES-CÔTE D'AZUR",
    "05": "PROVENCE-ALPES-CÔTE D'AZUR", "06": "PROVENCE-ALPES-CÔTE D'AZUR",
    "07": "AUVERGNE-RHÔNE-ALPES", "08": "GRAND-EST",
    "09": "OCCITANIE", "10": "GRAND-EST",
    "11": "OCCITANIE", "12": "OCCITANIE",
    "13": "PROVENCE-ALPES-CÔTE D'AZUR", "14": "NORMANDIE",
    "15": "AUVERGNE-RHÔNE-ALPES", "16": "NOUVELLE-AQUITAINE",
    "17": "NOUVELLE-AQUITAINE", "18": "CENTRE-VAL DE LOIRE",
    "19": "NOUVELLE-AQUITAINE", "2A": "CORSE",
    "2B": "CORSE", "21": "BOURGOGNE-FRANCHE-COMTÉ",
    "22": "BRETAGNE", "23": "NOUVELLE-AQUITAINE",
    "24": "NOUVELLE-AQUITAINE", "25": "BOURGOGNE-FRANCHE-COMTÉ",
    "26": "AUVERGNE-RHÔNE-ALPES", "27": "NORMANDIE",
    "28": "CENTRE-VAL DE LOIRE", "29": "BRETAGNE",
    "30": "OCCITANIE", "31": "OCCITANIE",
    "32": "OCCITANIE", "33": "NOUVELLE-AQUITAINE",
    "34": "OCCITANIE", "35": "BRETAGNE",
    "36": "CENTRE-VAL DE LOIRE", "37": "CENTRE-VAL DE LOIRE",
    "38": "AUVERGNE-RHÔNE-ALPES", "39": "BOURGOGNE-FRANCHE-COMTÉ",
    "40": "NOUVELLE-AQUITAINE", "41": "CENTRE-VAL DE LOIRE",
    "42": "AUVERGNE-RHÔNE-ALPES", "43": "AUVERGNE-RHÔNE-ALPES",
    "44": "PAYS DE LA LOIRE", "45": "CENTRE-VAL DE LOIRE",
    "46": "OCCITANIE", "47": "NOUVELLE-AQUITAINE",
    "48": "OCCITANIE", "49": "PAYS DE LA LOIRE",
    "50": "NORMANDIE", "51": "GRAND-EST",
    "52": "GRAND-EST", "53": "PAYS DE LA LOIRE",
    "54": "GRAND-EST", "55": "GRAND-EST",
    "56": "BRETAGNE", "57": "GRAND-EST",
    "58": "BOURGOGNE-FRANCHE-COMTÉ", "59": "HAUTS-DE-FRANCE",
    "60": "HAUTS-DE-FRANCE", "61": "NORMANDIE",
    "62": "HAUTS-DE-FRANCE", "63": "AUVERGNE-RHÔNE-ALPES",
    "64": "NOUVELLE-AQUITAINE", "65": "OCCITANIE",
    "66": "OCCITANIE", "67": "GRAND-EST",
    "68": "GRAND-EST", "69": "AUVERGNE-RHÔNE-ALPES",
    "70": "BOURGOGNE-FRANCHE-COMTÉ", "71": "BOURGOGNE-FRANCHE-COMTÉ",
    "72": "PAYS DE LA LOIRE", "73": "AUVERGNE-RHÔNE-ALPES",
    "74": "AUVERGNE-RHÔNE-ALPES", "75": "ÎLE-DE-FRANCE",
    "76": "NORMANDIE", "77": "ÎLE-DE-FRANCE",
    "78": "ÎLE-DE-FRANCE", "79": "NOUVELLE-AQUITAINE",
    "80": "HAUTS-DE-FRANCE", "81": "OCCITANIE",
    "82": "OCCITANIE", "83": "PROVENCE-ALPES-CÔTE D'AZUR",
    "84": "PROVENCE-ALPES-CÔTE D'AZUR", "85": "PAYS DE LA LOIRE",
    "86": "NOUVELLE-AQUITAINE", "87": "NOUVELLE-AQUITAINE",
    "88": "GRAND-EST", "89": "BOURGOGNE-FRANCHE-COMTÉ",
    "90": "BOURGOGNE-FRANCHE-COMTÉ", "91": "ÎLE-DE-FRANCE",
    "92": "ÎLE-DE-FRANCE", "93": "ÎLE-DE-FRANCE",
    "94": "ÎLE-DE-FRANCE", "95": "ÎLE-DE-FRANCE",
    "97": "OUTRE-MER", "98": "OUTRE-MER",
}

for rec in records:
    commune = rec["commune"]
    places = rec["places"]
    dept_num = rec["department_number"]

    if dept_num in DEPT_REGION_MAP:
        correct_region = DEPT_REGION_MAP[dept_num]
        if rec["region"] != correct_region:
            rec["region"] = correct_region

    is_port = rec["port_flag"] == 1
    if not is_port:
        if commune in PORT_ARTIFACT_COMMUNES:
            is_port = True
        elif any(kw.lower() in places.lower() for kw in PORT_KEYWORDS):
            is_port = True

    if is_port and rec["beach_flag"] == 1:
        rec["port_flag"] = 1
        rec["beach_flag"] = 0
        full_entry = f"{commune} {places}"
        embed_codes = re.findall(r"(\d{1,2}[A-Z]?)\.?\s*([A-Z\u00C9\u00C8\u00CA\u00CB\u00C0\u00C2\u00CE\u00CF\u00D4\u00DB\u00DC\u00C7\u2019' -]+)", full_entry)
        full_entry = dept_code_pattern.sub("", full_entry)
        if embed_codes:
            num, name = embed_codes[0]
            num = num.strip()
            if num != rec["department_number"]:
                rec["department_number"] = num
                rec["department_name"] = name.strip().rstrip("-").strip()
                if num in DEPT_REGION_MAP:
                    rec["region"] = DEPT_REGION_MAP[num]

        full_entry_nfc = unicodedata.normalize("NFC", full_entry)
        full_entry_haystack = full_entry_nfc.upper()
        for sep in ["CETTE ANNÉE", "PALMARÈS", "LÉGENDE"]:
            if sep in full_entry_haystack:
                idx = full_entry_haystack.index(sep)
                full_entry = full_entry_nfc[:idx]
                break
        for name in REGION_NAMES:
            full_entry = full_entry.replace(name, "")
        full_entry = re.sub(r"\s+", " ", full_entry).strip().rstrip(",").strip().rstrip(".").strip()
        full_entry = re.sub(r"\.[A-Z\u00C9\u00C8\u00CA\u00CB\u00C0\u00C2\u00CE\u00CF\u00D4\u00DB\u00DC\u00C7\u2019\u0027 -]+$", "", full_entry)
        full_entry = full_entry.strip().rstrip(".").strip()

        if full_entry:
            _town = full_entry.upper().strip()
            _port_prefixes = [
                "PORT DE PLAISANCE DE ", "PORT DE PLAISANCE ",
                "PORT FLUVIAL DES ", "PORT FLUVIAL DE ", "PORT FLUVIAL ",
                "PORT DE ", "PORT DES ", "PORT ",
                "HALTE FLUVIALE DE ", "HALTE FLUVIALE ",
                "HALTE DE ", "HALTE ",
                "RELAIS NAUTIQUE DE L'", "RELAIS NAUTIQUE DE ",
                "RELAIS NAUTIQUE D'", "RELAIS NAUTIQUE ",
                "BASSIN DE PLAISANCE DE ", "BASSIN DE PLAISANCE ",
                "BASSIN DE ", "BASSIN ",
                "NOUVEAU PORT DES ", "NOUVEAU PORT DE ", "NOUVEAU PORT ",
                "VIEUX PORT DES ", "VIEUX PORT DE ", "VIEUX PORT ",
                "STATION NAUTIQUE DE ",
                "CALANQUE DE ", "CNTL DE ", "SOCIÉTÉ NAUTIQUE DE ",
                "MARINA ",
            ]
            for _pfx in sorted(_port_prefixes, key=len, reverse=True):
                if _town.startswith(_pfx) and _town != _pfx:
                    _town = _town[len(_pfx):]
                    break
            _town = re.sub(r"\s*\(.*?\)", "", _town).strip()
            for _prep in ["DE LA ", "DE L'", "DU ", "DES ", "DE ", "D'"]:
                if _town.startswith(_prep):
                    _town = _town[len(_prep):]
                    break
            if _town:
                for _sep in [" DE ", " DES "]:
                    if _sep in _town:
                        _town = _town.rsplit(_sep, 1)[-1].strip()
                        break
                _person_port_map = {
                    "AJACCIO TINO ROSSI": "AJACCIO",
                    "CHARLES ORNANO": "AJACCIO",
                    "JEAN-BAPTISTE TOMI": "BASTIA",
                    "MICHEL ROTH": "LANEUVILLE-SUR-MEUSE",
                }
                if _town in _person_port_map:
                    _town = _person_port_map[_town]
                elif _town == "FRANCE":
                    _town = "NANCY"

            rec["commune"] = _town if _town else full_entry.upper()
            # Clean up PDF spacing artifacts in commune names (e.g., "T OULOUSE" -> "TOULOUSE")
            rec["commune"] = re.sub(r"(?<=[A-Z]) (?=[A-Z])", "", rec["commune"])
            rec["places"] = full_entry

    # Remove spurious single-letter + space patterns (like "T oulouse" -> "Toulouse")
    rec["commune"] = re.sub(r"(?<=[A-Z]) (?=[a-z])", "", rec["commune"])

# 7. Normalize curly apostrophe (U+2019) to straight apostrophe
for rec in records:
    for key in ("commune", "department_name", "region", "places"):
        rec[key] = rec[key].replace("\u2019", "'")

# 8. Merge records by commune + department
merged = {}
for rec in records:
    key = (rec["commune"], rec["department_number"])
    if key in merged:
        merged[key]["places"] = merged[key]["places"] + ", " + rec["places"]
        merged[key]["beach_flag"] = merged[key]["beach_flag"] or rec["beach_flag"]
        merged[key]["port_flag"] = merged[key]["port_flag"] or rec["port_flag"]
    else:
        merged[key] = dict(rec)

records = list(merged.values())

# 9. Remove records with empty places or missing commune
records[:] = [r for r in records if r["places"].strip() and r["commune"].strip()]

# 10. Manual department corrections
PORT_DEPT_CORRECTIONS = {
    ("NANCY", "52"): ("54", "MEURTHE-ET-MOSELLE"),
    ("FRANCE", "52"): ("54", "MEURTHE-ET-MOSELLE"),
}
for rec in records:
    key = (rec["commune"], rec["department_number"])
    if key in PORT_DEPT_CORRECTIONS:
        new_dept, new_dept_name = PORT_DEPT_CORRECTIONS[key]
        rec["department_number"] = new_dept
        rec["department_name"] = new_dept_name
        if new_dept in DEPT_REGION_MAP:
            rec["region"] = DEPT_REGION_MAP[new_dept]

# Re-write CSV
with open(OUTPUT_FILE, "w", newline="", encoding="utf-8") as f:
    fields = [
        "year", "commune", "department_name", "department_number",
        "region", "beach_flag", "port_flag", "places",
    ]
    writer = csv.DictWriter(f, fieldnames=fields)
    writer.writeheader()
    writer.writerows(records)

print(f"Cleanup complete! {len(records)} records written to {OUTPUT_FILE}")

In [ ]:
# Quick summary
print(f"Total records: {len(records)}")
print(f"Beach records: {sum(1 for r in records if r['beach_flag'] == 1)}")
print(f"Port records: {sum(1 for r in records if r['port_flag'] == 1)}")
print(f"\nSample beach records:")
count = 0
for r in records:
    if r['beach_flag'] == 1:
        print(f"  {r['commune']} ({r['department_number']} {r['department_name']}) - {r['places'][:60]}")
        count += 1
        if count >= 5:
            break
print(f"\nAll port records:")
for r in records:
    if r['port_flag'] == 1:
        print(f"  {r['commune']} ({r['department_number']} {r['department_name']}) - {r['places'][:80]}")